In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# ====================================================
# 1. KHỞI TẠO SPARK KẾT NỐI VỚI CLUSTER
# ====================================================
print("🚀 Đang khởi động Spark kết nối Cluster...")
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Cluster_Process_Articles_Multimodal") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Đã kết nối Spark thành công!")

🚀 Đang khởi động Spark kết nối Cluster...


26/03/25 16:00:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Đã kết nối Spark thành công!


In [2]:
# ====================================================
# 2. ĐỊNH NGHĨA ĐƯỜNG DẪN (HDFS)
# ====================================================
INPUT_FILE = "hdfs://namenode:9000/data/raw/articles.csv"
OUTPUT_PATH = "hdfs://namenode:9000/data/processed/articles_cleaned.parquet"

print(f"📥 Đang đọc file từ: {INPUT_FILE}")

try:
    # 1. Đọc dữ liệu (Dùng quote/escape để tránh lỗi dấu phẩy trong cột detail_desc)
    df = spark.read.csv(INPUT_FILE, header=True, inferSchema=True, quote='"', escape='"')

    # 2. Xử lý Article_ID (H&M ID phải có 10 chữ số, bắt đầu bằng 0)
    # Nếu đọc dạng số sẽ mất số 0, ta cần bù lại (lpad)
    df = df.withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

    # 3. Làm sạch và Gộp Text (Feature Engineering cho NLP/Multimodal)
    # Ta sẽ tạo một cột 'description' tổng hợp từ nhiều trường khác nhau
    print("✨ Đang làm sạch và chuẩn hóa dữ liệu...")
    
    # Danh sách các cột text muốn giữ lại
    text_cols = ["prod_name", "product_type_name", "product_group_name", 
                 "graphical_appearance_name", "colour_group_name", "detail_desc"]

    # Thay thế giá trị NULL bằng chuỗi rỗng để không làm hỏng hàm concat
    df = df.na.fill({c: "" for c in text_cols})

    df_processed = df.select(
        "article_id",
        "product_code",
        # Tạo cột text tổng hợp: Viết thường + Loại bỏ ký tự đặc biệt
        F.lower(
            F.regexp_replace(
                F.concat_ws(" ", *text_cols), 
                r"[^a-zA-Z0-9\s]", ""
            )
        ).alias("clean_description"),
        # Giữ lại các cột category quan trọng để lọc (Filter) sau này
        "index_name",
        "garment_group_name"
    )

    # 4. Kiểm tra dữ liệu sau xử lý
    print("✅ Xử lý hoàn tất! Xem trước dữ liệu:")
    df_processed.select("article_id", "clean_description").show(5, truncate=80)

    # ====================================================
    # 3. GHI KẾT QUẢ XUỐNG HDFS (PARQUET)
    # ====================================================
    print(f"📤 Đang ghi dữ liệu vào: {OUTPUT_PATH}")
    
    # Ghi dạng Parquet giúp tốc độ đọc sau này nhanh hơn 5-10 lần so với CSV
    df_processed.write.mode("overwrite").parquet(OUTPUT_PATH)
    
    print("🚀 Thành công rực rỡ! Hệ thống đã sẵn sàng cho bước tiếp theo.")

except Exception as e:
    print(f"❌ Lỗi rồi men ơi: {str(e)}")

# Đừng đóng spark vội nếu bạn còn muốn thao tác tiếp trên Terminal/Notebook
# spark.stop()

📥 Đang đọc file từ: hdfs://namenode:9000/data/raw/articles.csv


✨ Đang làm sạch và chuẩn hóa dữ liệu...
✅ Xử lý hoàn tất! Xem trước dữ liệu:


+----------+--------------------------------------------------------------------------------+
|article_id|                                                               clean_description|
+----------+--------------------------------------------------------------------------------+
|0108775015|strap top vest top garment upper body solid black jersey top with narrow shou...|
|0108775044|strap top vest top garment upper body solid white jersey top with narrow shou...|
|0108775051|strap top 1 vest top garment upper body stripe off white jersey top with narr...|
|0110065001|op tshirt idro bra underwear solid black microfibre tshirt bra with underwire...|
|0110065002|op tshirt idro bra underwear solid white microfibre tshirt bra with underwire...|
+----------+--------------------------------------------------------------------------------+
only showing top 5 rows

📤 Đang ghi dữ liệu vào: hdfs://namenode:9000/data/processed/articles_cleaned.parquet


🚀 Thành công rực rỡ! Hệ thống đã sẵn sàng cho bước tiếp theo.
